# Notebook 07: Benchmarking — PostgreSQL vs DuckDB

**Phase 6 — Head-to-Head on TPC-H Q1, Q3, Q5**

TPC-H is an industry-standard analytical benchmark.  The three queries chosen
here cover a spectrum of OLAP shapes:

| Query | Shape | Why chosen |
|:---|:---|:---|
| Q1 | Pure aggregation, no joins, 6M rows | Maximises DuckDB's columnar advantage |
| Q3 | 3-table join + aggregation | Tests join pipeline performance |
| Q5 | 6-table star join + GROUP BY nation | Complex OLAP shape, all major dimension tables |

**Methodology**
- 3 timed runs per query per engine; **median** taken (middle value is immune to
  a single OS scheduling spike or cache miss that would skew the mean)
- PostgreSQL: psycopg2, indexes in place, `cursor.execute()` with full fetch
- DuckDB: in-process, Parquet files, `duck.execute().fetchall()`
- Both engines execute on the same machine — relative speedup is what matters

---

## Prerequisites

- Notebook 01 run (Parquet files present, PostgreSQL seeded)
- Indexes from Notebook 04 in place (or run the setup cell below)


In [ ]:
import pathlib
import statistics
import sys
import time

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
import psycopg2

ROOT = pathlib.Path.cwd()
while not (ROOT / 'pyproject.toml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from config import settings

PARQUET_DIR = ROOT / "data" / "parquet"
SQL_DIR     = ROOT / "sql" / "benchmarking"

# ── PostgreSQL connection ────────────────────────────────────────────────────
pg = psycopg2.connect(settings.dsn)
pg.autocommit = True

# ── DuckDB connection ────────────────────────────────────────────────────────
duck = duckdb.connect()

# Register all TPC-H Parquet files as views (same names as PostgreSQL tables)
for t in ["region", "nation", "supplier", "part", "partsupp",
          "customer", "orders", "lineitem"]:
    duck.execute(
        f"CREATE VIEW {t} AS SELECT * FROM read_parquet('{PARQUET_DIR / t}.parquet')"
    )

print(f"PostgreSQL: {settings.POSTGRES_HOST}:{settings.POSTGRES_PORT}/{settings.POSTGRES_DB}")
print(f"DuckDB:     {duckdb.__version__}  —  Parquet dir: {PARQUET_DIR}")


def load_section(filename: str, section: str) -> str:
    text = (SQL_DIR / filename).read_text(encoding="utf-8")
    blocks: dict[str, str] = {}
    current: str | None = None
    acc: list[str] = []
    for line in text.splitlines():
        if line.startswith("-- §"):
            if current is not None:
                blocks[current] = "\n".join(acc).strip()
            current = line[4:].strip()
            acc = []
        else:
            acc.append(line)
    if current is not None:
        blocks[current] = "\n".join(acc).strip()
    if section not in blocks:
        raise KeyError(f"Section '{section}' not found in {filename}. Available: {list(blocks)}")
    return blocks[section]


RUNS = 3


def bench_pg(sql: str) -> float:
    """Run sql against PostgreSQL RUNS times; return median seconds."""
    times = []
    cur = pg.cursor()
    for _ in range(RUNS):
        t0 = time.perf_counter()
        cur.execute(sql)
        cur.fetchall()
        times.append(time.perf_counter() - t0)
    cur.close()
    return statistics.median(times)


def bench_duck(sql: str) -> float:
    """Run sql against DuckDB RUNS times; return median seconds."""
    times = []
    for _ in range(RUNS):
        t0 = time.perf_counter()
        duck.execute(sql).fetchall()
        times.append(time.perf_counter() - t0)
    return statistics.median(times)


In [ ]:
# Ensure the indexes from Phase 3 exist — idempotent (IF NOT EXISTS).
with pg.cursor() as cur:
    cur.execute("""
        CREATE INDEX IF NOT EXISTS idx_orders_orderdate
            ON orders (o_orderdate);
        CREATE INDEX IF NOT EXISTS idx_lineitem_shipdate
            ON lineitem (l_shipdate);
    """)
print("Indexes verified.")


---

## Q1 — Pricing Summary Report

**Shape:** aggregate-only, no joins, scans all 6M lineitem rows.  
**Why DuckDB wins here:** only 4 of 16 columns are needed — the columnar layout
reads ~25% of the data a row-store must touch.  Vectorised SIMD arithmetic
processes 1024 values per instruction batch.


In [ ]:
q1 = load_section("q1_pricing_summary.sql", "q1")
print(q1)


In [ ]:
# Warm up both engines (first run may load OS file cache / JIT)
pg_cur = pg.cursor()
pg_cur.execute(q1); pg_cur.fetchall(); pg_cur.close()
duck.execute(q1).fetchall()

pg_q1   = bench_pg(q1)
duck_q1 = bench_duck(q1)

print(f"PostgreSQL  Q1: {pg_q1*1000:7.1f} ms")
print(f"DuckDB      Q1: {duck_q1*1000:7.1f} ms")
print(f"Speedup:        {pg_q1/duck_q1:.1f}x")


---

## Q3 — Shipping Priority

**Shape:** 3-table join (customer × orders × lineitem), partial aggregation,
`LIMIT 10` on revenue.  
**Trade-off:** hash joins benefit from columnar when the build side fits in
CPU cache; PostgreSQL's row-level MVCC has lower overhead for small join keys.


In [ ]:
q3 = load_section("q3_shipping_priority.sql", "q3")
print(q3)


In [ ]:
pg_cur = pg.cursor()
pg_cur.execute(q3); pg_cur.fetchall(); pg_cur.close()
duck.execute(q3).fetchall()

pg_q3   = bench_pg(q3)
duck_q3 = bench_duck(q3)

print(f"PostgreSQL  Q3: {pg_q3*1000:7.1f} ms")
print(f"DuckDB      Q3: {duck_q3*1000:7.1f} ms")
print(f"Speedup:        {pg_q3/duck_q3:.1f}x")


---

## Q5 — Local Supplier Volume

**Shape:** 6-table star join (region, nation, customer, orders, lineitem,
supplier), GROUP BY nation.  Every major dimension table is in play.  
**Why this tests join strategy:** the supplier join requires matching on two
columns simultaneously (`l_suppkey = s_suppkey AND s_nationkey = c.c_nationkey`),
which forces a hash join build on a composite key.


In [ ]:
q5 = load_section("q5_local_supplier_volume.sql", "q5")
print(q5)


In [ ]:
pg_cur = pg.cursor()
pg_cur.execute(q5); pg_cur.fetchall(); pg_cur.close()
duck.execute(q5).fetchall()

pg_q5   = bench_pg(q5)
duck_q5 = bench_duck(q5)

print(f"PostgreSQL  Q5: {pg_q5*1000:7.1f} ms")
print(f"DuckDB      Q5: {duck_q5*1000:7.1f} ms")
print(f"Speedup:        {pg_q5/duck_q5:.1f}x")


In [ ]:
queries   = ["Q1 (agg only)", "Q3 (3-table join)", "Q5 (6-table join)"]
pg_times  = [pg_q1,   pg_q3,   pg_q5]
duck_times = [duck_q1, duck_q3, duck_q5]

x = range(len(queries))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_pg   = ax.bar([i - width/2 for i in x], [t*1000 for t in pg_times],
                   width, label="PostgreSQL", color="C0", alpha=0.85)
bars_duck = ax.bar([i + width/2 for i in x], [t*1000 for t in duck_times],
                   width, label="DuckDB",     color="C2", alpha=0.85)

# Speedup annotations above each pair
for i, (pg_t, dk_t) in enumerate(zip(pg_times, duck_times)):
    speedup = pg_t / dk_t
    ax.text(i, max(pg_t, dk_t)*1000 + 20, f"{speedup:.1f}x",
            ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_xticks(list(x))
ax.set_xticklabels(queries)
ax.set_ylabel("Median latency (ms)  —  lower is better")
ax.set_title("TPC-H Q1 / Q3 / Q5: PostgreSQL vs DuckDB  (sf=1, 3-run median)")
ax.legend()
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f} ms"))
plt.tight_layout()
plt.show()


In [ ]:
# Summary table
df_results = pd.DataFrame({
    "Query":       queries,
    "PostgreSQL (ms)": [f"{t*1000:.1f}" for t in pg_times],
    "DuckDB (ms)":     [f"{t*1000:.1f}" for t in duck_times],
    "Speedup":         [f"{p/d:.1f}x"   for p, d in zip(pg_times, duck_times)],
})
df_results


---

## Conclusion

**What the numbers show:**

- **Q1 (pure aggregation)** — DuckDB wins by the largest margin.  Scanning 4 of
  16 columns from a columnar Parquet file, then processing them in 1024-wide
  SIMD batches, is the exact workload columnar engines are designed for.
  PostgreSQL must read every column in every row even though only 4 are needed.

- **Q3 & Q5 (joins)** — DuckDB still wins, but the gap narrows.  Both engines use
  hash joins; the advantage shifts from pure I/O savings toward DuckDB's
  vectorised aggregation on the probe side.  PostgreSQL's MVCC overhead is
  minimal here since there are no concurrent writes.

**When to use each engine:**

| Workload | Engine |
|:---|:---|
| Analytical aggregations, Parquet, local dev, CI | **DuckDB** |
| Row-level CRUD, ACID transactions, concurrency | **PostgreSQL** |
| Hybrid (operational + analytical over same data) | **Both** — `postgres_scan` |

**The lakehouse pattern:** PostgreSQL handles writes and enforces consistency.
DuckDB queries the warehouse layer (Parquet / object storage) analytically without
touching the OLTP pool.  Both databases have a permanent role in a modern data
stack — they are complements, not competitors.
